# Research Paper Midterm Check-In - Demonstration Notebook

Author: James Williams (jwill436)


## Imports and Setup

In [1]:
from pathlib import Path
import torch
import sys
import torch.nn.functional as F
from tqdm.auto import tqdm
sys.path.append(str(Path.cwd().parent))
from modules.data_loading.mpd.reader import MPDConfig
from modules.data_loading.mpd.make_datasets import make_mpd_loaders
from modules.models.decode_only_transformer import ModelConfig, DecodeOnlyTransformer
from modules.coherence.cooccurence import build_cooccurrence_store, build_dense_similarity_matrix, sequential_coherence_scores_fast
from modules.coherence.losses import combined_ce_coherence_loss


## Data Loading Pipeline

I demonstrate the data loading pipeline that I've implemented and do some sanity checks on the output, by looking at a small sample of 128 training playlists.

In [2]:
config = MPDConfig(
    data_dir=Path.cwd().parent / 'datasets' / 'MPD' / 'data',
    min_track_freq=1,
    min_playlist_len=2,
    max_seq_len=64,
    seed=10,
    max_train_playlists=128
)


In [3]:
train_loader, val_loader, test_loader, vocab, train_sequences = make_mpd_loaders(
    config=config,
    batch_size=8,
    num_workers=0
)

print('Vocab size:', len(vocab))
print('Train batches:', len(train_loader))
print('Val batches:', len(val_loader))
print('Test batches:', len(test_loader))


Train playlists collected: 128
Val playlists collected: 12
Test playlists collected: 12


  0%|          | 0/128 [00:00<?, ?it/s]

Vocab size: 6645


  0%|          | 0/128 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

Train sequences kept: 128
Val sequences kept: 12
Test sequences kept: 12
Vocab size: 6645
Train batches: 16
Val batches: 2
Test batches: 2


In [4]:
batch = next(iter(train_loader))

for k, v in batch.items():
    print(k, v.shape)
    

input_ids torch.Size([8, 63])
labels torch.Size([8, 63])
attention_mask torch.Size([8, 63])


In [5]:
print(batch['input_ids'][1])
print(batch['labels'][1])
print(batch['attention_mask'][1])


tensor([   1, 4723, 6171,   35, 4482,   58, 2310, 2636, 6227, 4233,  684, 2378,
        6389,  201, 4000, 3150, 6246, 3809, 3926, 1684,  955,  414, 3449, 6357,
         313, 2281, 2799, 1208, 4255,  961, 2109, 1468, 5939, 2102,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0])
tensor([4723, 6171,   35, 4482,   58, 2310, 2636, 6227, 4233,  684, 2378, 6389,
         201, 4000, 3150, 6246, 3809, 3926, 1684,  955,  414, 3449, 6357,  313,
        2281, 2799, 1208, 4255,  961, 2109, 1468, 5939, 2102,    2, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [6]:
seq = batch['input_ids'][1]

tokens = [vocab.decode_token(int(i)) for i in seq if i != vocab.pad_idx]

tokens[:20]


['[BOS]',
 'spotify:track:5Zb94ke9RQDjpa4yKW2aoA',
 'spotify:track:7DoBn08dqKS9MmtZiZChVD',
 'spotify:track:02G8w6L5nlaK7M0e2Ytinu',
 'spotify:track:5HDlh6UhT3AMQs935wE1qr',
 'spotify:track:03YuyrltrfOOtysn0HBvax',
 'spotify:track:2jIu30Vg6ox7sK0Oy2DK2Q',
 'spotify:track:384N4lsAziKXZ1MlDFTUAU',
 'spotify:track:7Il449tBVu9kGcHsEt3HH1',
 'spotify:track:4y6P6eic8hhTcFqtaV5ftn',
 'spotify:track:0oEbCQvG8L9vzh6yoxrPJx',
 'spotify:track:2onPMbsXumicHnkJRewhhI',
 'spotify:track:7i325B6qZFdkne07FRsWlk',
 'spotify:track:0EDkumRSuGgoKeO0BInZhf',
 'spotify:track:4i9XWt2x2lHWFnRWmAY1nX',
 'spotify:track:3kcdygid5z1yxQEMQDSBLS',
 'spotify:track:7KFUj2MunvS4cbauF04r6w',
 'spotify:track:4VrAOvoshwVVc6wZQjuwu6',
 'spotify:track:4dIWbsYRqqTcLKf4bVU4kk',
 'spotify:track:24TMaKkgOU8w3HSIhqN643']

## Baseline Model Pipeline

Research suggested that decoder-only transformers were the best architecture for the task. I wrote a class to implement models, and I demonstrate that I can overfit it on the small subset, ie that it is capable of learning from the data.

In [7]:
model_config = ModelConfig(
    num_tracks=len(vocab),
    d_model=128,
    n_heads=4,
    n_layers=2,
    d_ff=256,
    dropout=0.0,
    max_seq_len=config.max_seq_len,
    pad_idx=vocab.pad_idx,
)

model = DecodeOnlyTransformer(model_config)

batch = next(iter(train_loader))
logits = model(batch['input_ids'], batch['attention_mask'])

print('input_ids:', batch['input_ids'].shape)
print('labels:', batch['labels'].shape)
print('logits:', logits.shape)


input_ids: torch.Size([8, 63])
labels: torch.Size([8, 63])
logits: torch.Size([8, 63, 6645])


In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    total_tokens = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            labels.view(-1),
            ignore_index=-100,
        )
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            valid_tokens = (labels != -100).sum().item()
            total_loss += loss.item() * valid_tokens
            total_tokens += valid_tokens
    avg_loss = total_loss / max(total_tokens, 1)
    print(f'Epoch {epoch + 1:02d} | train token CE: {avg_loss:.4f}')
    

Epoch 01 | train token CE: 82.3707
Epoch 02 | train token CE: 57.9638
Epoch 03 | train token CE: 37.3641
Epoch 04 | train token CE: 31.6058
Epoch 05 | train token CE: 28.9526
Epoch 06 | train token CE: 27.1755
Epoch 07 | train token CE: 25.7623
Epoch 08 | train token CE: 24.5474
Epoch 09 | train token CE: 23.4768
Epoch 10 | train token CE: 22.5209
Epoch 11 | train token CE: 21.6950
Epoch 12 | train token CE: 20.9413
Epoch 13 | train token CE: 20.2701
Epoch 14 | train token CE: 19.6345
Epoch 15 | train token CE: 19.0658
Epoch 16 | train token CE: 18.5215
Epoch 17 | train token CE: 18.0261
Epoch 18 | train token CE: 17.5802
Epoch 19 | train token CE: 17.1499
Epoch 20 | train token CE: 16.7432


In [9]:
model.eval()

batch = next(iter(train_loader))
input_ids = batch['input_ids'][:1].to(device)
attention_mask = batch['attention_mask'][:1].to(device)
labels = batch['labels'][:1].to(device)

with torch.no_grad():
    logits = model(input_ids, attention_mask)
    preds = logits.argmax(dim=-1)

inp = input_ids[0].tolist()
lab = labels[0].tolist()
prd = preds[0].tolist()

for i in range(min(15, len(inp))):
    in_tok = vocab.decode_token(inp[i])
    y_tok = vocab.decode_token(lab[i]) if lab[i] != -100 else '[IGN]'
    p_tok = vocab.decode_token(prd[i])
    print(f'{i:02d}')
    print(f'in: {in_tok}')
    print(f'y : {y_tok}')
    print(f'p : {p_tok}')
    print()
    

00
in: [BOS]
y : spotify:track:4xUcvrovqXbwt57StASoQ4
p : spotify:track:3mNecsYFb6LQg7822DPXCP

01
in: spotify:track:4xUcvrovqXbwt57StASoQ4
y : spotify:track:436yrzQWA32vb1sTZKXg9r
p : spotify:track:6vTtMyCg96xwpoIBws9K0Q

02
in: spotify:track:436yrzQWA32vb1sTZKXg9r
y : spotify:track:55NWdAwu1muDwDnSzGojjA
p : spotify:track:5N1itNLaYtifU8YVbg9SBQ

03
in: spotify:track:55NWdAwu1muDwDnSzGojjA
y : spotify:track:4YMQXzscifAREG0a7KNGhB
p : spotify:track:55NWdAwu1muDwDnSzGojjA

04
in: spotify:track:4YMQXzscifAREG0a7KNGhB
y : spotify:track:5z4iT44mMHyZozsTFy4A51
p : spotify:track:2YhapA5E1qTZ1Ykw9Y86ql

05
in: spotify:track:5z4iT44mMHyZozsTFy4A51
y : spotify:track:5H9osUMcTbPs1Hzo6tIHSa
p : spotify:track:4hd4x0iaXAQ2pIcyUgHB7h

06
in: spotify:track:5H9osUMcTbPs1Hzo6tIHSa
y : spotify:track:2w3ScXudq4aD3K5HFO5xvx
p : spotify:track:4eLSCSELtKxZwXnFbNLXT5

07
in: spotify:track:2w3ScXudq4aD3K5HFO5xvx
y : spotify:track:1udKn1oNKYQSQ9OmiIWCMu
p : spotify:track:5XwIdRXvHTtxH5BmBOSCCM

08
in: spotify:

## Incorporating Coherence

The primary metric is track-to-track coherence, or sequential coherence, where similarity is defined by common playlist membership; tracks which co-occur in more playlists are more similar, and songs which never co-occur are perfectly dissimilar.

I demonstrate that the metrics work; I don't focus on their actual values at all.

In [10]:
store = build_cooccurrence_store(
    sequences=train_sequences,
    vocab=vocab,
)

similarity_matrix = build_dense_similarity_matrix(
    store=store,
    device=device,
)

print('Similarity matrix shape:', similarity_matrix.shape)

Similarity matrix shape: torch.Size([6645, 6645])


In [11]:
model_config = ModelConfig(
    num_tracks=len(vocab),
    d_model=128,
    n_heads=4,
    n_layers=2,
    d_ff=256,
    dropout=0.0,
    max_seq_len=config.max_seq_len,
    pad_idx=vocab.pad_idx,
    tie_weights=True,
)

model = DecodeOnlyTransformer(model_config).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

num_epochs = 10
coherence_weight = 0.1

for epoch in tqdm(range(num_epochs)):
    model.train()

    total_loss_sum = 0.0
    total_ce_sum = 0.0
    total_coh_sum = 0.0
    total_tokens = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)

        coh_scores = sequential_coherence_scores_fast(
            input_ids=input_ids,
            similarity_matrix=similarity_matrix,
            attention_mask=attention_mask,
        )

        loss_dict = combined_ce_coherence_loss(
            logits=logits,
            labels=labels,
            coherence_scores=coh_scores,
            attention_mask=attention_mask,
            ce_weight=1.0,
            coherence_weight=coherence_weight,
            coherence_temperature=1.0,
        )

        loss = loss_dict['loss']
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            valid_tokens = (labels != -100).sum().item()
            total_tokens += valid_tokens
            total_loss_sum += float(loss_dict['loss']) * valid_tokens
            total_ce_sum += float(loss_dict['ce_loss']) * valid_tokens
            total_coh_sum += float(loss_dict['coherence_loss']) * valid_tokens

    print(f'Epoch {epoch + 1:02d}')
    print(f'total: {total_loss_sum / max(total_tokens, 1):.4f}')
    print(f'cross-entropy: {total_ce_sum / max(total_tokens, 1):.4f}')
    print(f'coherence: {total_coh_sum / max(total_tokens, 1):.4f}')
    print()


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01
total: 84.4741
cross-entropy: 84.5717
coherence: -0.9761

Epoch 02
total: 64.4065
cross-entropy: 64.4958
coherence: -0.8932

Epoch 03
total: 37.8325
cross-entropy: 37.8583
coherence: -0.2581

Epoch 04
total: 31.0503
cross-entropy: 31.0602
coherence: -0.0989

Epoch 05
total: 28.4659
cross-entropy: 28.4749
coherence: -0.0897

Epoch 06
total: 26.7856
cross-entropy: 26.7945
coherence: -0.0887

Epoch 07
total: 25.4545
cross-entropy: 25.4635
coherence: -0.0896

Epoch 08
total: 24.2851
cross-entropy: 24.2940
coherence: -0.0892

Epoch 09
total: 23.2984
cross-entropy: 23.3072
coherence: -0.0878

Epoch 10
total: 22.4279
cross-entropy: 22.4364
coherence: -0.0852

